# Fire distribution across Australia

## Accessing Wildfire Data via API

In [ ]:
# import necessary libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium


In [ ]:
# 1.
# access api url

## satellite: VIIRS SNPP NRT 
## area: 'world' = entire world 
## day range: '1' = data of one day
## date: None = most recent available data, so today's data

MAP_KEY = '4899a992545cbeb46f9fd0b6a025ef17'
area_url ='https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_NRT/world/1' # warum gehen 1, 3 oder 5 tage aber ab 8 oder so nicht mehr??

# 2.
# read in the data from URL

df_area = pd.read_csv(area_url)

# 3.
# have a first glimpse at the data

df_area.head(5)
df_area.shape

## Cleaning and Rearranging Data

### Filter for Data only within Australia

In [ ]:
# define a bounding box that contains only the area of Australia based on its WGS84 coordinates

coords = [112, -44, 154, -9]

df_aus = df_area[(df_area['longitude'] >= coords[0]) & (df_area['latitude'] >= coords[1]) & (df_area['longitude'] <= coords[2]) & (df_area['latitude'] <= coords[3])].copy()
df_aus.shape
df_aus.head(20)
df_aus.tail()

### Filter for required Timeframe

In [ ]:
# 1. 
# combine the acq_date and acq_time column to one acq_datetime column and set it to an active time format with pandas function to_datetime

## acq_date is a string in the format YYYY-MM_DD, 
## while acq_time is an integer in Greenwich Mean Time (e.g. 603 meaning 6:03), 
## so it needs to be converted to string too (with astype(str)),
## fill it up to 4 numbers with zeros, so that all times have the same length (with str.zfill(4), e.g. 603 -> 0603)
## and save it as the format '%Y-%m-%d %H%M'

df_aus['acq_datetime'] = pd.to_datetime(df_aus['acq_date'] + ' ' + df_aus['acq_time'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
df_aus.head()

print (f'Australia GMT timezone datetime value range: {df_aus['acq_datetime'].min()} to {df_aus['acq_datetime'].max()}')

# 2.
# convert GMT into local time?


### Converting raw coordinates into geometries

In [ ]:
# the projection EPSG:9473 is used for Australia, as it is recommended for national mapping

# convert latitude, longitude values into point geometry and set crs (since no crs extisting yet) with crs="EPSG:9473" to EPSG:9473

gdf_aus = gpd.GeoDataFrame(
    df_aus, geometry=gpd.points_from_xy(df_aus.longitude, df_aus.latitude), crs="EPSG:9473")
print(gdf_aus.crs)
gdf_aus.head()

## Calculating means etc?

## Visualise it and create interactive Map

In [ ]:
aus_map = folium.Map(
    location=[42, 147],
    zoom_start=13,
    tiles="CartoDB Positron",  # A clean, light basemap
)

# Display the map in the notebook
aus_map